# Vehicle Sales Prediction — Complete ML Pipeline
**Dataset**: CarDekho (`car data.csv`) &nbsp;|&nbsp; **Target**: `Selling_Price` (Lakh INR)

---
## Table of Contents
1. [Imports](#section-1)
2. [Load Dataset](#section-2)
3. [Data Cleaning](#section-3)
4. [Feature Engineering](#section-4)
5. [EDA — Exploratory Data Analysis](#section-5)
6. [Correlation Heatmap](#section-6)
7. [Train/Test Split](#section-7)
8. [Preprocessing Pipeline](#section-8)
9. [Baseline Models](#section-9)
10. [Model Evaluation](#section-10)
11. [Model Selection](#section-11)
12. [Hyperparameter Tuning](#section-12)
13. [Feature Importance](#section-13)
14. [Actual vs Predicted](#section-14)
15. [Save Production Model](#section-15)


## Section 1 — Imports <a id='section-1'></a>

In [ ]:
# Standard library
import json
import warnings
from datetime import datetime
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# ML
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# ── Project paths ─────────────────────────────────────────────────────────────
BASE_DIR      = Path('..')          # notebooks/ -> ml/
DATA_PATH     = BASE_DIR / 'data' / 'car data.csv'
MODEL_DIR     = BASE_DIR / 'models'
ARTIFACT_DIR  = BASE_DIR / 'artifacts'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
RANDOM_STATE  = 42
TEST_SIZE     = 0.20
CURRENT_YEAR  = datetime.now().year   # dynamic — never hardcoded

print(f'Libraries loaded | Current year: {CURRENT_YEAR}')
print(f'Data path: {DATA_PATH.resolve()}')


## Section 2 — Load Dataset <a id='section-2'></a>

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()


In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)


In [ ]:
print('Descriptive statistics:')
df.describe().round(3)


## Section 3 — Data Cleaning <a id='section-3'></a>

### 3a. Missing values

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing.to_string())
print(f'\nTotal missing: {missing.sum()}')


### 3b. Duplicate rows

In [ ]:
n_before = len(df)
df.drop_duplicates(inplace=True)
n_after = len(df)
print(f'Duplicates removed: {n_before - n_after}  ({n_before} -> {n_after} rows)')


### 3c. Categorical value inspection & normalisation

In [ ]:
cat_cols = ['Fuel_Type', 'Seller_Type', 'Transmission']
print('Before normalisation:')
for col in cat_cols:
    print(f'  {col}: {df[col].unique().tolist()}')

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

print('\nAfter normalisation:')
for col in cat_cols:
    print(f'  {col}: {sorted(df[col].unique().tolist())}')


### 3d. Numerical sanity checks

In [ ]:
print('Owner unique:', sorted(df['Owner'].unique().tolist()))
print('Year  range :', df['Year'].min(), '-', df['Year'].max())
print('Selling_Price range:', df['Selling_Price'].min(), '-', df['Selling_Price'].max())
print('Present_Price range:', df['Present_Price'].min(), '-', df['Present_Price'].max())
print('Kms_Driven range   :', df['Kms_Driven'].min(), '-', df['Kms_Driven'].max())

n0 = len(df)
df = df[df['Selling_Price'] > 0]
df = df[df['Present_Price'] > 0]
df = df[df['Kms_Driven'] > 0]
df = df[(df['Year'] >= 1990) & (df['Year'] <= CURRENT_YEAR)]
df = df[df['Owner'].isin([0, 1, 2, 3])]
df.reset_index(drop=True, inplace=True)
print(f'\nRows after sanity checks: {n0} -> {len(df)}')


### 3e. Car_Name note

In [ ]:
# Car_Name is fully anonymised in this dataset (Car_0 .. Car_300).
# Brand extraction is not possible from the available data.
# Present_Price (ex-showroom) already encodes brand-level pricing information.
print('Car_Name unique values:', df['Car_Name'].nunique())
print('Sample:', df['Car_Name'].head(5).tolist())
print()
print('NOTE: Car_Name is anonymised. Brand cannot be extracted.')
print('      Car_Name will be dropped. Present_Price serves as brand proxy.')


## Section 4 — Feature Engineering <a id='section-4'></a>

In [ ]:
# Car_Age: use dynamic current year (never hardcoded)
df['Car_Age'] = CURRENT_YEAR - df['Year']
print(f'Car_Age = {CURRENT_YEAR} - Year')
print(f'Car_Age range: {df["Car_Age"].min()} - {df["Car_Age"].max()} years')

# Drop columns not used for modelling
df.drop(columns=['Car_Name', 'Year'], inplace=True)

print(f'\nFinal columns ({len(df.columns)}): {df.columns.tolist()}')
df.head()


## Section 5 — Exploratory Data Analysis <a id='section-5'></a>

### 5a. Selling Price distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram with KDE
sns.histplot(df['Selling_Price'], kde=True, ax=axes[0], color='steelblue', bins=25)
axes[0].set_title('Selling Price Distribution')
axes[0].set_xlabel('Selling Price (Lakh INR)')
axes[0].set_ylabel('Count')

# Box plot
sns.boxplot(y=df['Selling_Price'], ax=axes[1], color='steelblue')
axes[1].set_title('Selling Price Box Plot')
axes[1].set_ylabel('Selling Price (Lakh INR)')

plt.suptitle('Target Variable: Selling Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(df['Selling_Price'].describe().round(3).to_string())


### 5b. Selling Price vs Fuel Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(x='Fuel_Type', y='Selling_Price', data=df,
            palette='Set2', ax=axes[0])
axes[0].set_title('Selling Price vs Fuel Type')
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('Selling Price (Lakh INR)')

sns.violinplot(x='Fuel_Type', y='Selling_Price', data=df,
               palette='Set2', ax=axes[1], inner='box')
axes[1].set_title('Selling Price Distribution by Fuel Type')
axes[1].set_xlabel('Fuel Type')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


### 5c. Selling Price vs Car Age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
scatter = ax.scatter(df['Car_Age'], df['Selling_Price'],
                     c=df['Selling_Price'], cmap='viridis', alpha=0.7, s=60)
plt.colorbar(scatter, ax=ax, label='Selling Price')
ax.set_xlabel('Car Age (years)')
ax.set_ylabel('Selling Price (Lakh INR)')
ax.set_title('Selling Price vs Car Age')

sns.boxplot(x='Transmission', y='Selling_Price', data=df,
            palette='Set1', ax=axes[1])
axes[1].set_title('Selling Price vs Transmission')
axes[1].set_xlabel('Transmission')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


### 5d. Seller Type and Owner analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(x='Seller_Type', y='Selling_Price', data=df,
            palette='pastel', ax=axes[0])
axes[0].set_title('Selling Price vs Seller Type')
axes[0].set_xlabel('Seller Type')
axes[0].set_ylabel('Selling Price (Lakh INR)')

owner_avg = df.groupby('Owner')['Selling_Price'].mean().reset_index()
sns.barplot(x='Owner', y='Selling_Price', data=owner_avg,
            palette='Blues_d', ax=axes[1])
axes[1].set_title('Average Selling Price by Number of Owners')
axes[1].set_xlabel('Number of Previous Owners')
axes[1].set_ylabel('Avg Selling Price (Lakh INR)')

plt.tight_layout()
plt.show()


### 5e. Present Price vs Selling Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.scatter(df['Present_Price'], df['Selling_Price'],
           alpha=0.65, color='coral', edgecolors='white', s=60)
ax.set_xlabel('Present Price / Ex-showroom (Lakh INR)')
ax.set_ylabel('Selling Price (Lakh INR)')
ax.set_title('Present Price vs Selling Price')

ax2 = axes[1]
ax2.scatter(df['Kms_Driven'], df['Selling_Price'],
            alpha=0.55, color='teal', edgecolors='white', s=60)
ax2.set_xlabel('Kilometres Driven')
ax2.set_ylabel('Selling Price (Lakh INR)')
ax2.set_title('Kilometres Driven vs Selling Price')

plt.tight_layout()
plt.show()


## Section 6 — Correlation Heatmap <a id='section-6'></a>

In [ ]:
# Select only numeric columns for correlation
num_df = df.select_dtypes(include=['number'])
print('Numeric columns:', num_df.columns.tolist())

plt.figure(figsize=(9, 7))
corrmat = num_df.corr()
mask = np.triu(np.ones_like(corrmat, dtype=bool))
sns.heatmap(
    corrmat, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', linewidths=0.5,
    vmin=-1, vmax=1, center=0,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Matrix (Numeric Features)', fontsize=13)
plt.tight_layout()
plt.show()


## Section 7 — Train/Test Split <a id='section-7'></a>

In [ ]:
X = df.drop(columns=['Selling_Price'])
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f'Training set  : {X_train.shape}  ({len(X_train)} rows)')
print(f'Test set      : {X_test.shape}  ({len(X_test)} rows)')
print(f'Split ratio   : {1-TEST_SIZE:.0%} / {TEST_SIZE:.0%}')
print(f'random_state  : {RANDOM_STATE}')
print()
print('Feature columns:', X.columns.tolist())
print('Target         : Selling_Price')


## Section 8 — Preprocessing Pipeline <a id='section-8'></a>

A single `ColumnTransformer` handles all preprocessing:

- **Numerical** (`Present_Price`, `Kms_Driven`, `Car_Age`, `Owner`): `StandardScaler`
- **Categorical** (`Fuel_Type`, `Seller_Type`, `Transmission`): `OneHotEncoder(drop='first')`

This preprocessor is embedded inside every `Pipeline`, so the same transformation
used during training is automatically applied during inference — no leakage.


In [ ]:
NUMERICAL_FEATURES   = ['Present_Price', 'Kms_Driven', 'Car_Age', 'Owner']
CATEGORICAL_FEATURES = ['Fuel_Type', 'Seller_Type', 'Transmission']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERICAL_FEATURES),
    ('cat', OneHotEncoder(drop='first', sparse_output=False,
                          handle_unknown='ignore'), CATEGORICAL_FEATURES),
])

print('Preprocessor built:')
print(f'  Numerical   : {NUMERICAL_FEATURES}')
print(f'  Categorical : {CATEGORICAL_FEATURES}')
print()
print('NOTE: preprocessor is only fit on X_train inside each Pipeline.fit().')
print('      The same transforms are used automatically at predict time.')


## Section 9 — Baseline Models <a id='section-9'></a>

In [ ]:
def evaluate_pipeline(pipe, X_tr, y_tr, X_te, y_te):
    pipe.fit(X_tr, y_tr)
    p = pipe.predict(X_te)
    return (
        round(mean_absolute_error(y_te, p), 4),
        round(float(np.sqrt(mean_squared_error(y_te, p))), 4),
        round(r2_score(y_te, p), 4),
        p
    )

baseline_defs = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
    'GradientBoostingRegressor': GradientBoostingRegressor(
        n_estimators=100, random_state=RANDOM_STATE),
}

baseline_results = []
baseline_fitted  = {}
baseline_preds   = {}

for name, mdl in baseline_defs.items():
    pipe = Pipeline([('pre', preprocessor), ('mdl', mdl)])
    mae, rmse, r2, preds = evaluate_pipeline(
        pipe, X_train, y_train, X_test, y_test)
    baseline_results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    baseline_fitted[name]  = pipe
    baseline_preds[name]   = preds
    print(f'  {name:30s}  MAE={mae}  RMSE={rmse}  R2={r2}')


## Section 10 — Model Evaluation <a id='section-10'></a>

In [ ]:
results_df = pd.DataFrame(baseline_results)
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)

print('=== Model Comparison ===')
print(results_df.to_string(index=False))


In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'R2']
colors  = ['coral', 'steelblue', 'seagreen']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'R2'))
    sns.barplot(data=sorted_df, x='Model', y=metric, palette=[color]*3, ax=ax)
    ax.set_title(f'{metric} (lower is better)' if metric != 'R2' else 'R2 (higher is better)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Baseline Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 11 — Model Selection <a id='section-11'></a>

In [ ]:
# Programmatic selection — highest R2 on test set
best_idx    = results_df['R2'].idxmax()
best_row    = results_df.loc[best_idx]
best_name   = best_row['Model']
best_pipe   = baseline_fitted[best_name]

print(f'Selection criterion : highest R2 on test set')
print(f'Selected model      : {best_name}')
print(f'  MAE  = {best_row["MAE"]}')
print(f'  RMSE = {best_row["RMSE"]}')
print(f'  R2   = {best_row["R2"]}')


## Section 12 — Hyperparameter Tuning <a id='section-12'></a>

### 12a. Tune Random Forest

In [ ]:
rf_param_dist = {
    'mdl__n_estimators':      [100, 200, 300],
    'mdl__max_depth':         [None, 10, 20],
    'mdl__min_samples_split': [2, 5],
    'mdl__min_samples_leaf':  [1, 2],
    'mdl__max_features':      ['sqrt', 'log2'],
}

rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('mdl', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
])

rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_dist, n_iter=12, cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE, n_jobs=1, verbose=0
)
rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
p_rf = rf_best.predict(X_test)
mae_rf  = round(mean_absolute_error(y_test, p_rf), 4)
rmse_rf = round(float(np.sqrt(mean_squared_error(y_test, p_rf))), 4)
r2_rf   = round(r2_score(y_test, p_rf), 4)

print('Best RF params:', rf_search.best_params_)
print(f'Tuned RF  :  MAE={mae_rf}  RMSE={rmse_rf}  R2={r2_rf}')


### 12b. Tune Gradient Boosting

In [ ]:
gb_param_dist = {
    'mdl__n_estimators':  [100, 200, 300],
    'mdl__learning_rate': [0.05, 0.1, 0.15],
    'mdl__max_depth':     [3, 4, 5],
    'mdl__subsample':     [0.8, 1.0],
}

gb_pipe = Pipeline([
    ('pre', preprocessor),
    ('mdl', GradientBoostingRegressor(random_state=RANDOM_STATE)),
])

gb_search = RandomizedSearchCV(
    gb_pipe, gb_param_dist, n_iter=12, cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE, n_jobs=1, verbose=0
)
gb_search.fit(X_train, y_train)

gb_best = gb_search.best_estimator_
p_gb = gb_best.predict(X_test)
mae_gb  = round(mean_absolute_error(y_test, p_gb), 4)
rmse_gb = round(float(np.sqrt(mean_squared_error(y_test, p_gb))), 4)
r2_gb   = round(r2_score(y_test, p_gb), 4)

print('Best GB params:', gb_search.best_params_)
print(f'Tuned GB  :  MAE={mae_gb}  RMSE={rmse_gb}  R2={r2_gb}')


### 12c. Tuning comparison

In [ ]:
tuning_df = pd.DataFrame([
    {'Model': 'RF Baseline',  'MAE': best_row['MAE'] if best_name == 'RandomForestRegressor' else None,
     'RMSE': best_row['RMSE'] if best_name == 'RandomForestRegressor' else None,
     'R2': best_row['R2'] if best_name == 'RandomForestRegressor' else None},
    {'Model': 'RF Tuned',   'MAE': mae_rf,  'RMSE': rmse_rf,  'R2': r2_rf},
    {'Model': 'GB Baseline', 'MAE': results_df[results_df['Model']=='GradientBoostingRegressor']['MAE'].values[0],
     'RMSE': results_df[results_df['Model']=='GradientBoostingRegressor']['RMSE'].values[0],
     'R2'  : results_df[results_df['Model']=='GradientBoostingRegressor']['R2'].values[0]},
    {'Model': 'GB Tuned',  'MAE': mae_gb,  'RMSE': rmse_gb,  'R2': r2_gb},
]).dropna(subset=['R2'])

print(tuning_df.to_string(index=False))


## Section 13 — Feature Importance <a id='section-13'></a>

In [ ]:
# Programmatic production model selection: best tuned R2
if r2_rf >= r2_gb:
    prod_pipeline = rf_best
    prod_name     = 'RandomForestRegressor (Tuned)'
    prod_mae, prod_rmse, prod_r2 = mae_rf, rmse_rf, r2_rf
    prod_params   = rf_search.best_params_
else:
    prod_pipeline = gb_best
    prod_name     = 'GradientBoostingRegressor (Tuned)'
    prod_mae, prod_rmse, prod_r2 = mae_gb, rmse_gb, r2_gb
    prod_params   = gb_search.best_params_

# Compare against baseline winner
if float(best_row['R2']) > prod_r2:
    prod_pipeline = best_pipe
    prod_name     = best_name + ' (Baseline)'
    prod_mae      = float(best_row['MAE'])
    prod_rmse     = float(best_row['RMSE'])
    prod_r2       = float(best_row['R2'])
    prod_params   = {}

print(f'=== PRODUCTION MODEL ===')
print(f'Name : {prod_name}')
print(f'MAE  : {prod_mae}')
print(f'RMSE : {prod_rmse}')
print(f'R2   : {prod_r2}')


In [ ]:
# Extract feature importances from the production model
pre_step = prod_pipeline.named_steps['pre']
mdl_step = prod_pipeline.named_steps['mdl']

if hasattr(mdl_step, 'feature_importances_'):
    cat_enc   = pre_step.named_transformers_['cat']
    cat_names = cat_enc.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
    all_names = NUMERICAL_FEATURES + cat_names

    fi_df = pd.DataFrame({
        'Feature':    all_names,
        'Importance': mdl_step.feature_importances_,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)

    print(fi_df.to_string(index=False))

    plt.figure(figsize=(9, 5))
    sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
    plt.title(f'Feature Importances — {prod_name}', fontsize=12)
    plt.xlabel('Importance Score')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / 'feature_importance.png', dpi=120)
    plt.show()
else:
    print(f'{prod_name} does not expose feature_importances_.')
    print('Use permutation importance as an alternative.')


## Section 14 — Actual vs Predicted <a id='section-14'></a>

In [ ]:
prod_preds = prod_pipeline.predict(X_test)
residuals  = y_test.values - prod_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: actual vs predicted
ax = axes[0]
ax.scatter(y_test, prod_preds, alpha=0.7, color='steelblue',
           edgecolors='white', s=65, label='Predictions')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual Selling Price (Lakh INR)')
ax.set_ylabel('Predicted Selling Price (Lakh INR)')
ax.set_title('Actual vs Predicted')
ax.legend()

# Residuals
sns.histplot(residuals, kde=True, ax=axes[1], color='coral', bins=20)
axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual  (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution')

plt.suptitle(f'Production Model: {prod_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'actual_vs_predicted.png', dpi=120)
plt.show()

print(f'Residuals  mean : {residuals.mean():.4f}')
print(f'Residuals  std  : {residuals.std():.4f}')


## Section 15 — Save Production Model <a id='section-15'></a>

In [ ]:
# ── Save complete pipeline (preprocessor + model) ─────────────────────────
MODEL_PATH    = MODEL_DIR    / 'car_price_model.pkl'
METADATA_PATH = ARTIFACT_DIR / 'model_metadata.json'

joblib.dump(prod_pipeline, MODEL_PATH)
print(f'Pipeline saved -> {MODEL_PATH}')

# ── Metadata ───────────────────────────────────────────────────────────────
metadata = {
    'model_name':             prod_name,
    'feature_columns':        X_train.columns.tolist(),
    'numerical_features':     NUMERICAL_FEATURES,
    'categorical_features':   CATEGORICAL_FEATURES,
    'target':                 'Selling_Price',
    'training_rows':          int(X_train.shape[0]),
    'testing_rows':           int(X_test.shape[0]),
    'mae':                    prod_mae,
    'rmse':                   prod_rmse,
    'r2':                     prod_r2,
    'available_categories': {
        'Fuel_Type':    ['Petrol', 'Diesel', 'Cng'],
        'Seller_Type':  ['Dealer', 'Individual'],
        'Transmission': ['Manual', 'Automatic'],
        'Owner':        [0, 1, 2, 3],
    },
    'hyperparams':            {str(k): str(v) for k, v in prod_params.items()},
    'current_year_used':      CURRENT_YEAR,
    'training_date':          datetime.now().isoformat(),
    'dataset_file':           'car data.csv',
    'dataset_rows':           int(df.shape[0]),
    'dataset_cols':           9,
    'baseline_comparison':    results_df.to_dict(orient='records'),
}

with open(METADATA_PATH, 'w', encoding='utf-8') as fh:
    json.dump(metadata, fh, indent=2)

print(f'Metadata saved -> {METADATA_PATH}')
print()
print(json.dumps(metadata, indent=2))


---
## Summary

| Item | Value |
|---|---|
| Production model | See `prod_name` above |
| Saved pipeline | `ml/models/car_price_model.pkl` |
| Metadata | `ml/artifacts/model_metadata.json` |

The saved `Pipeline` object contains both the `ColumnTransformer` preprocessor and the
tuned estimator, ready to be loaded directly by the FastAPI backend.
